# Week 1: Real Quotes and a Budgeted Trip

**120-minute lab.** Start from the validated five-adult SIN-NRT-Tokyo scenario in `config/trip_scenario.py`: 20-23 March 2027, three nights, five single rooms, SGD 7,000 budget. Use real Nuitee REST searches. Do not book or pay.

Teacher setup and REST walkthrough: 25 min. Student tasks: occupancy 10 min; normalization 35 min; selection 10 min; plan and budget 15 min. Tests and discussion: 25 min.

Complete the same four tasks in `src/travel_challenge/week1/trip_planner.py`. Scenario edits are settings, not additional exercises. Week 2 reuses these exact functions. No A2A, MCP, x402 or AP2 service is needed this week. Functions intentionally raise `NotImplementedError` until you implement them.

Flight quotes cover all configured adults and both legs. Hotel offers cover all configured rooms for the complete stay. Never multiply either total again. All normalized prices are integer cents with an explicit currency.

## Environment and Travel Setup (25 min)

**Private environment:** copy the root `.env.example` to `.env`, then replace the placeholder with your Nuitee Sandbox API key. Do not overwrite an existing `.env`, paste keys into notebooks, or commit credentials. Codespaces/operating-system variables take precedence. The key is loaded only when a Nuitee request runs.

**Customer requirements:** open `src/travel_challenge/config/trip_scenario.py`. Its `TRIP` dictionary is the only place to edit traveller count, airports, hotel city, dates, room adult counts and budget. The default is the validated five-adult Tokyo trip. `[1, 1, 1, 1, 1]` means five single rooms; `[2, 2, 1]` means three rooms for five adults. Room occupants must add up to travellers.

The teacher's `get_trip_request()` copies these settings and converts the SGD budget to integer cents. It supports adults, SGD and hotel cities in Japan; use airport/city/date combinations supported by Nuitee. Searches use economy flights and SG nationality. Provider availability and room limits still apply; unsupported requests fail visibly, with no invented fallback.

Use the prepared Python 3.11 environment and its notebook kernel. Run the next cell to reload the scenario. Read the two request bodies in `week1/nuitee_rest.py`; the real HTTP call remains visible.

In [ ]:
import sys
import subprocess
import importlib
from pathlib import Path
from travel_challenge.week1 import nuitee_rest, trip_planner
from travel_challenge.config import trip_scenario

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
importlib.reload(trip_scenario)
request = trip_planner.get_trip_request()
print(request)
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week1.py', '-m', 'scaffold', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, check.stderr

## Task 1: Room Occupancy (10 min)

Implement `hotel_occupancies(adults_per_room)` in `week1/trip_planner.py`. Receive the room adult-count list from the request and return one Nuitee occupancy dictionary per room. The teacher's setup already checks that occupants add up to the configured traveller count.

Use a basic loop and dictionary; no room-allocation algorithm, children or traveller categories. The default remains five single-adult rooms. Change only the room list in `config/trip_scenario.py` to try a supported shared-room arrangement. After a scenario edit, rerun the setup cell first; after a function edit, rerun the next cell to reload it.

In [ ]:
importlib.reload(trip_planner)
occupancies = trip_planner.hotel_occupancies(request['adults_per_room'])
assert len(occupancies) == len(request['adults_per_room'])
for index, room in enumerate(occupancies):
    assert room['adults'] == request['adults_per_room'][index]
print(occupancies)

## Query Real Offers

The teacher supplies these HTTP functions. Read the request bodies in `nuitee_rest.py`, then make two read-only searches using the shared scenario request.

Flights use all configured adults and two legs. Hotels use the configured room occupancies. Both use SGD. A hotel `offerRetailRate` is the whole requested stay for all requested rooms, not a nightly or per-room amount.

No hotel-details enrichment is needed. The hotel rates response already includes hotel names. An empty response means no available quotes; do not invent a fallback. No API key is needed until this request cell; a missing key produces a short configuration error.

In [ ]:
raw_flights = nuitee_rest.search_flights(request)
raw_hotels = nuitee_rest.search_hotels(request, occupancies)
assert raw_flights.get('data'), 'No flights returned; inspect the provider response.'
assert raw_hotels.get('data'), 'No hotels returned; inspect the provider response.'
journey = raw_flights['data'][0]['journeys'][0]
hotel_offer = raw_hotels['data'][0]['roomTypes'][0]
print('All-traveller round-trip pricing:', journey['cheapestOffer']['pricing']['display'])
print('All-room full-stay pricing:', hotel_offer['offerRetailRate'])
print('Sandbox marker:', raw_hotels.get('sandbox', 'not supplied'))

## Task 2: Normalize Both Responses (35 min)

Edit `src/travel_challenge/week1/trip_planner.py::normalize_flights` and `normalize_hotels`. The basic loops are supplied. Worked example: `quote_id = offer['offerId']` extracts the supplier flight offer ID; it is not an airline or hotel ID. Complete the remaining extraction and append each quote dictionary to `quotes`.

Flight paths: `data[].journeys[].cheapestOffer` contains `offerId` and `pricing.display.total/currency`; duration is `journey['totalDuration']['minutes']`. Required output: `id`, `amount_cents`, `currency`, `duration_minutes`.

Hotel paths: `data[].roomTypes[]` contains `offerId` and `offerRetailRate.amount/currency`. Find the name in `hotels[]` whose `id` equals `data[].hotelId`. Required output: `id`, `merchant_id`, `merchant_name`, `amount_cents`, `currency`, `rooms`. Derive `rooms` from the length of `request['adults_per_room']`, not the number of travellers.

Use the imported `Decimal` with `Decimal(str(amount))`, round supplier quotes to cents with Decimal's default `ROUND_HALF_EVEN`, then convert to `int`. The teacher setup rejects fractional-cent budgets instead of rounding them. Reject a currency different from the request. Totals already cover all travellers or rooms and the full stay: never multiply again. Keep the cheapest-price rule; no extra ranking or hotel-details API.

In [ ]:
importlib.reload(trip_planner)
flights = trip_planner.normalize_flights(raw_flights, request)
hotels = trip_planner.normalize_hotels(raw_hotels, request)
print('Flight candidates:', len(flights), 'Hotel candidates:', len(hotels))
print(flights[0])
print(hotels[0])

## Task 3: Choose One Quote (10 min)

Implement `choose_option(options)` in `trip_planner.py`: return the lowest `amount_cents` option, or raise `ValueError` when no offers are available. Use the same rule for flights and hotels. No ranking weights or custom optimizer.

In [ ]:
importlib.reload(trip_planner)
selected_flight = trip_planner.choose_option(flights)
selected_hotel = trip_planner.choose_option(hotels)
print('Selected flight:', selected_flight)
print('Selected hotel:', selected_hotel)

## Task 4: Assemble the Budgeted Plan (15 min)

Implement `make_plan(request, flight, hotel)` in `trip_planner.py`. Preserve the request and both selected quotes. Require matching currency, sum the two integer-cent totals exactly once, and compare with `budget_cents`.

Return `request`, `flight`, `hotel`, `total_cents` and `within_budget`. A plan over budget is a valid result marked `False`, not an excuse to remove a hotel or invent a cheaper quote. Replanning belongs to Week 4.

In [ ]:
import json
importlib.reload(trip_planner)
plan = trip_planner.make_plan(request, selected_flight, selected_hotel)
print('Total SGD:', plan['total_cents'] / 100)
print('Budget SGD:', request['budget_cents'] / 100)
print('Within budget:', plan['within_budget'])
print('Supplier quote IDs:', plan['flight']['id'], plan['hotel']['id'])
(ROOT / 'trip_plan.json').write_text(json.dumps(plan, indent=2), encoding='utf-8')
print('Plan exported for Weeks 3-4; no booking or payment was made.')

In [ ]:
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week1.py', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, 'Complete the four tasks in trip_planner.py. ' + check.stderr

## Tests and Discussion (25 min)

Teacher-only checks: `python -m pytest tests/test_week1.py -m scaffold -q`. Student checks: replace `scaffold` with `student`. All Week 1 checks: omit `-m`. Run commands from the repository root.

Teacher checks should pass immediately. Student checks intentionally fail with `NotImplementedError` until the four tasks are done. They cover cent conversion, quote IDs, currency rejection, all-room pricing, empty offers and the exact budget boundary. They do not prove live availability.

Explain why a quote ID differs from a hotel ID, and why a nightly or per-person price cannot be added directly to the trip budget. Show one plan below budget and one above it.

The ignored `trip_plan.json` is the only cross-week data file. Week 2 overwrites it with the agent-built plan; Weeks 3-4 use its selected hotel. No reservation has been made.